# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
version_before_bad = DeltaTable(table_path).version()
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    bad_write_blocked = False
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    bad_write_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")
# A blocked write must not create a commit: the table version is unchanged.
version_after_bad = DeltaTable(table_path).version()
print(f"Table version before/after the bad write: v{version_before_bad} → v{version_after_bad}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
Table version before/after the bad write: v0 → v0


## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 4b. Evidence — `_delta_log/` after two commits

List the commit files and print the actions inside the second commit
(the schema-merge append). Each line of a commit JSON is one action.

In [6]:
import json
from pathlib import Path as _P

log_files = sorted(_P(table_path).glob("_delta_log/*.json"))
for f in log_files:
    print(f"  {f.name}  ({f.stat().st_size} B)")

print()
print(f"Actions in {log_files[-1].name}:")
for line in log_files[-1].read_text(encoding="utf-8").splitlines():
    action = json.loads(line)
    kind = next(iter(action))
    body = action[kind]
    if kind == "metaData":
        fields = [fld["name"] for fld in json.loads(body["schemaString"])["fields"]]
        print(f"  metaData  → schema fields = {fields}")
    elif kind == "add":
        print(f"  add       → {body['path'][:60]}  size={body['size']} B  stats={body.get('stats')}")
    elif kind == "commitInfo":
        print(f"  commitInfo→ operation={body.get('operation')}  params={body.get('operationParameters')}")
    else:
        print(f"  {kind:<9} → {json.dumps(body)[:100]}")

  00000000000000000000.json  (1409 B)
  00000000000000000001.json  (1465 B)

Actions in 00000000000000000001.json:
  commitInfo→ operation=WRITE  params={'mode': 'Append'}
  metaData  → schema fields = ['id', 'name', 'age', 'city', 'tier']
  add       → part-00000-5fc733c4-38ba-4b5f-9535-3831d9b53f0a-c000.snappy.  size=1548 B  stats={"numRecords":1,"minValues":{"name":"dan","age":28,"city":"Hue","id":4,"tier":"premium"},"maxValues":{"id":4,"tier":"premium","age":28,"city":"Hue","name":"dan"},"nullCount":{"age":0,"name":0,"id":0,"city":0,"tier":0}}


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [7]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The schema-enforcement flag below is set by the bad-write cell itself
(originally a hardcoded `True`), and it also requires that the blocked
write did not create a new table version.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": bad_write_blocked and version_after_bad == version_before_bad,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits


  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## Giải thích kết quả (NB1)

* **Transaction log.** Sau hai lần ghi thành công, `_delta_log/` có đúng hai commit
  `00000000000000000000.json` (v0 — `WRITE`, 3 dòng) và `00000000000000000001.json`
  (v1 — append có `schema_mode="merge"`). Mỗi dòng trong commit là một *action*.
  Commit v1 in ở trên có 3 action: `commitInfo` (operation `WRITE`, mode `Append`),
  `metaData` (schema mới 5 field, đã có `tier`) và `add` (file Parquet mới kèm `stats`:
  numRecords, min/max, nullCount theo cột — chính là dữ liệu NB2 dùng để skip file).
  Bảng Delta = thư mục Parquet + log này; reader dựng lại trạng thái hiện tại bằng
  cách replay các action theo thứ tự version.
* **Schema enforcement.** Ghi `age="thirty"` vào cột `Int64` bị chặn ngay khi writer
  cast dữ liệu (`Cannot cast string 'thirty' to value of Int64 type`). Quan trọng hơn
  dòng PASS: version bảng **không đổi** sau lần ghi lỗi (v0 → v0), tức không có commit
  nửa vời nào — đó là tính *atomic* của ACID. Cờ kiểm tra cuối notebook giờ lấy từ
  chính cell này thay cho giá trị `True` hardcode trong bản gốc.
* **Schema evolution là opt-in.** Chỉ khi truyền `schema_mode="merge"` thì cột `tier`
  mới được thêm vào `metaData` của commit v1. Ba dòng cũ không bị rewrite — đọc lên
  với `tier = null`, nên DuckDB thấy 2 nhóm: `premium` (1) và `NULL` (3).
  Nếu không opt-in, cùng lần ghi đó sẽ bị từ chối như lần ghi sai kiểu.